# 04 — Baseline historique requête → SKU

**Objectif :** mémoriser les clics des requêtes connues et compléter avec la popularité.

**Entrées :** split du notebook 03.  
**Sorties :** mapping local et métriques.  
**Dépendance :** notebook 03.  
**Temps estimé :** moins d'une minute.  
**Ressources :** CPU uniquement.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd
import yaml


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import evaluate_rankings

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
TOP_K = int(CONFIG["project"]["top_k"])
PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
ARTIFACTS_DIR = ROOT / CONFIG["paths"]["artifacts"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

fit_frame = pd.read_csv(PROCESSED_DIR / "fit_clicks.csv", dtype={"sku": str})
validation = pd.read_csv(PROCESSED_DIR / "validation_clicks.csv", dtype={"sku": str})

## Construction du mapping

In [ ]:
global_ranking = fit_frame["sku"].value_counts().index.astype(str).tolist()
all_skus = pd.concat([fit_frame["sku"], validation["sku"]]).drop_duplicates().astype(str).tolist()
global_ranking.extend(sku for sku in all_skus if sku not in global_ranking)

click_history: dict[str, list[str]] = {}
for query_key, group in fit_frame.groupby("query_key"):
    click_history[str(query_key)] = group["sku"].value_counts().index.astype(str).tolist()


def recommend(query_key: str, k: int = TOP_K) -> list[str]:
    ranking: list[str] = []
    for sku in [*click_history.get(query_key, []), *global_ranking]:
        if sku not in ranking:
            ranking.append(sku)
        if len(ranking) == k:
            break
    return ranking


sample_known = next(iter(click_history))
print(sample_known, "→", recommend(sample_known))
assert len(recommend(sample_known)) == TOP_K
assert len(set(recommend(sample_known))) == TOP_K

## Évaluation stricte sur requêtes jamais vues

In [ ]:
actual_by_query = validation.groupby("query_key")["sku"].agg(lambda values: set(map(str, values)))
predictions = [recommend(str(query)) for query in actual_by_query.index]
metrics = evaluate_rankings(actual_by_query.tolist(), predictions, TOP_K)
seen_rate = sum(query in click_history for query in actual_by_query.index) / len(actual_by_query)

report = {
    "model": "query_click_history_with_popularity_fallback",
    "split": "group_shuffle_by_query_key",
    "known_query_rate": seen_rate,
    "history_queries": len(click_history),
    **metrics,
}

(ARTIFACTS_DIR / "click_history.json").write_text(
    json.dumps(
        {"click_history": click_history, "global_ranking": global_ranking},
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
(REPORTS_DIR / "metrics_query_click.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Interprétation

Avec le split strict, le taux de requêtes connues doit être nul : la performance provient donc
du fallback. En production, les requêtes déjà observées bénéficient immédiatement de l'historique.